In [ ]:
!pip -q install -U "transformers>=4.48.0" datasets accelerate evaluate scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 53.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is inco

In [ ]:
import os
import json
import math
import random
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

MODEL_NAME = "answerdotai/ModernBERT-large"
MAX_LENGTH = 2048
PREFIX_TRUNCATION_SAFETY_TOKENS = 32

SEED = 42

TRAIN_SIZE = 0.80
VAL_SIZE = 0.10
TEST_SIZE = 0.10

NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8

USE_CLASS_WEIGHTS = True

MAX_TOTAL_EXAMPLES = None

OUTPUT_DIR = "/content/modernbert_joint_verifier_outputs"
SAVE_DIR = "/content/modernbert_joint_verifier_best"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. This notebook is intended for an H100 GPU runtime.")

GPU: NVIDIA H100 80GB HBM3


In [ ]:
NEXT_STEP_PATH = ""
SOLUTION_PATH = ""

def find_file_under_content(filename: str):
    matches = list(Path("/content").rglob(filename))
    return str(matches[0]) if matches else ""

if not NEXT_STEP_PATH:
    NEXT_STEP_PATH = find_file_under_content("next_step_verification.jsonl")

if not SOLUTION_PATH:
    SOLUTION_PATH = find_file_under_content("solution_verification.jsonl")

if not NEXT_STEP_PATH or not SOLUTION_PATH:
    from google.colab import files
    print("Could not find both JSONL files automatically. Please upload:")
    print("- next_step_verification.jsonl")
    print("- solution_verification.jsonl")
    uploaded = files.upload()

    uploaded_names = list(uploaded.keys())
    for name in uploaded_names:
        lower = name.lower()
        if "next" in lower and "step" in lower and lower.endswith(".jsonl"):
            NEXT_STEP_PATH = f"/content/{name}"
        if "solution" in lower and lower.endswith(".jsonl"):
            SOLUTION_PATH = f"/content/{name}"

print("NEXT_STEP_PATH:", NEXT_STEP_PATH)
print("SOLUTION_PATH:", SOLUTION_PATH)

assert NEXT_STEP_PATH and Path(NEXT_STEP_PATH).exists(), "Could not find next_step_verification.jsonl"
assert SOLUTION_PATH and Path(SOLUTION_PATH).exists(), "Could not find solution_verification.jsonl"

Could not find both JSONL files automatically. Please upload:
- next_step_verification.jsonl
- solution_verification.jsonl


Saving next_step_verification.jsonl to next_step_verification.jsonl
Saving solution_verification.jsonl to solution_verification.jsonl
NEXT_STEP_PATH: /content/next_step_verification.jsonl
SOLUTION_PATH: /content/solution_verification.jsonl


In [ ]:
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Skipping malformed JSON in {path}, line {line_num}: {e}")
    return rows

next_rows_raw = read_jsonl(NEXT_STEP_PATH)
solution_rows_raw = read_jsonl(SOLUTION_PATH)

print("Raw next-step rows:", len(next_rows_raw))
print("Raw solution rows:", len(solution_rows_raw))

print("\nExample next-step keys:", sorted(next_rows_raw[0].keys()) if next_rows_raw else None)
print("Example solution keys:", sorted(solution_rows_raw[0].keys()) if solution_rows_raw else None)

Raw next-step rows: 26256
Raw solution rows: 905

Example next-step keys: ['instruction', 'label', 'prefix_steps', 'problem', 'raw_rating', 'step_index', 'target_step']
Example solution keys: ['instruction', 'label', 'problem', 'solution']


In [ ]:
def normalize_text(x: Any) -> str:
    if x is None:
        return ""
    return str(x).strip()

def normalize_prefix_steps(prefix_steps: Any) -> List[str]:
    if prefix_steps is None:
        return []
    if isinstance(prefix_steps, list):
        return [normalize_text(s) for s in prefix_steps if normalize_text(s)]
    return [normalize_text(prefix_steps)] if normalize_text(prefix_steps) else []

def normalize_label(x: Any) -> int:
    y = int(x)
    assert y in (0, 1), f"Label must be 0 or 1, got {x}"
    return y

def make_next_step_example(row: Dict[str, Any]) -> Dict[str, Any]:
    problem = normalize_text(row.get("problem"))
    prefix_steps = normalize_prefix_steps(row.get("prefix_steps"))
    target_step = normalize_text(row.get("target_step"))
    instruction = normalize_text(row.get("instruction")) or (
        "Verify whether the target step is mathematically valid given the problem and the previous steps."
    )

    return {
        "task_type": "next_step",
        "instruction": instruction,
        "problem": problem,
        "prefix_steps": prefix_steps,
        "target_step": target_step,
        "solution": "",
        "label": normalize_label(row.get("label")),
        "group_problem": problem,
        "raw_rating": row.get("raw_rating"),
        "step_index": row.get("step_index"),
    }

def make_solution_example(row: Dict[str, Any]) -> Dict[str, Any]:
    problem = normalize_text(row.get("problem"))
    solution = normalize_text(row.get("solution"))
    instruction = normalize_text(row.get("instruction")) or "Verify whether the following full solution is correct."

    return {
        "task_type": "solution",
        "instruction": instruction,
        "problem": problem,
        "prefix_steps": [],
        "target_step": "",
        "solution": solution,
        "label": normalize_label(row.get("label")),
        "group_problem": problem,
        "raw_rating": None,
        "step_index": None,
    }

next_examples = [
    make_next_step_example(r)
    for r in next_rows_raw
    if normalize_text(r.get("problem")) and normalize_text(r.get("target_step")) and r.get("label") is not None
]

solution_examples = [
    make_solution_example(r)
    for r in solution_rows_raw
    if normalize_text(r.get("problem")) and normalize_text(r.get("solution")) and r.get("label") is not None
]

all_examples = next_examples + solution_examples

if MAX_TOTAL_EXAMPLES is not None and len(all_examples) > MAX_TOTAL_EXAMPLES:
    rng = random.Random(SEED)
    all_examples = rng.sample(all_examples, MAX_TOTAL_EXAMPLES)

print("Normalized next-step examples:", len(next_examples))
print("Normalized solution examples:", len(solution_examples))
print("Total examples used:", len(all_examples))

print("\nOverall label counts:", Counter(x["label"] for x in all_examples))
print("Task counts:", Counter(x["task_type"] for x in all_examples))

for task in ["next_step", "solution"]:
    task_examples = [x for x in all_examples if x["task_type"] == task]
    print(f"{task} label counts:", Counter(x["label"] for x in task_examples))

Normalized next-step examples: 26256
Normalized solution examples: 905
Total examples used: 27161

Overall label counts: Counter({1: 20634, 0: 6527})
Task counts: Counter({'next_step': 26256, 'solution': 905})
next_step label counts: Counter({1: 20176, 0: 6080})
solution label counts: Counter({1: 458, 0: 447})


In [ ]:
def count_tokens(text: str, tokenizer, add_special_tokens: bool = False) -> int:
    return len(tokenizer(text, add_special_tokens=add_special_tokens, truncation=False)["input_ids"])

def format_solution_example(example: Dict[str, Any]) -> str:
    instruction = example["instruction"].strip()
    problem = example["problem"].strip()
    solution = example["solution"].strip()

    return (
        "Instruction:\n"
        f"{instruction}\n\n"
        "Task: full-solution verification\n\n"
        "Problem:\n"
        f"{problem}\n\n"
        "Solution:\n"
        f"{solution}\n\n"
        "Binary label meaning: 1 = correct solution, 0 = incorrect solution."
    )

def format_next_step_without_prefix_truncation(example: Dict[str, Any]) -> str:
    instruction = example["instruction"].strip()
    problem = example["problem"].strip()
    target_step = example["target_step"].strip()
    prefix_steps = example.get("prefix_steps", [])

    if prefix_steps:
        prefix_text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(prefix_steps))
    else:
        prefix_text = "(No previous steps.)"

    return (
        "Instruction:\n"
        f"{instruction}\n\n"
        "Task: next-step verification\n\n"
        "Target step:\n"
        f"{target_step}\n\n"
        "Problem:\n"
        f"{problem}\n\n"
        "Previous steps:\n"
        f"{prefix_text}\n\n"
        "Binary label meaning: 1 = valid/correct step, 0 = invalid/incorrect step."
    )

def format_next_step_with_recent_prefix(
    example: Dict[str, Any],
    tokenizer,
    max_length: int,
    safety_tokens: int = PREFIX_TRUNCATION_SAFETY_TOKENS,
) -> Tuple[str, Dict[str, int]]:
    instruction = example["instruction"].strip()
    problem = example["problem"].strip()
    target_step = example["target_step"].strip()
    prefix_steps = example.get("prefix_steps", [])

    header = (
        "Instruction:\n"
        f"{instruction}\n\n"
        "Task: next-step verification\n\n"
        "Target step:\n"
        f"{target_step}\n\n"
        "Problem:\n"
        f"{problem}\n\n"
        "Previous steps, with the most recent available context preserved:\n"
    )
    footer = "\nBinary label meaning: 1 = valid/correct step, 0 = invalid/incorrect step."

    if not prefix_steps:
        text = header + "(No previous steps.)\n" + footer
        info = {
            "num_prefix_steps_original": 0,
            "num_prefix_steps_kept": 0,
            "num_prefix_steps_dropped": 0,
            "core_too_long": 0,
        }
        return text, info

    core_text = header + footer
    core_len = count_tokens(core_text, tokenizer, add_special_tokens=True)
    available_for_prefix = max_length - core_len - safety_tokens

    if available_for_prefix <= 0:
        text = header + "[All previous steps omitted because the problem and target step already fill the context window.]\n" + footer
        info = {
            "num_prefix_steps_original": len(prefix_steps),
            "num_prefix_steps_kept": 0,
            "num_prefix_steps_dropped": len(prefix_steps),
            "core_too_long": 1,
        }
        return text, info

    kept_reversed = []
    used = 0

    for original_idx in range(len(prefix_steps) - 1, -1, -1):
        step = prefix_steps[original_idx]
        step_text = f"{original_idx + 1}. {step}\n"
        step_len = count_tokens(step_text, tokenizer, add_special_tokens=False)
        if used + step_len > available_for_prefix:
            break
        kept_reversed.append(step_text)
        used += step_len

    kept_steps = list(reversed(kept_reversed))
    kept_count = len(kept_steps)
    dropped_count = len(prefix_steps) - kept_count

    if kept_steps:
        prefix_text = "".join(kept_steps)
    else:
        prefix_text = "[All previous steps omitted because they do not fit in the context window.]\n"

    if dropped_count > 0:
        prefix_text = f"[Omitted {dropped_count} earliest previous step(s) due to context length.]\n" + prefix_text

    text = header + prefix_text + footer
    info = {
        "num_prefix_steps_original": len(prefix_steps),
        "num_prefix_steps_kept": kept_count,
        "num_prefix_steps_dropped": dropped_count,
        "core_too_long": 0,
    }
    return text, info

def format_example(
    example: Dict[str, Any],
    tokenizer=None,
    max_length: int = MAX_LENGTH,
    smart_truncate_next_step_prefix: bool = True,
) -> Tuple[str, Dict[str, int]]:
    task_type = example["task_type"]

    if task_type == "solution":
        text = format_solution_example(example)
        info = {
            "num_prefix_steps_original": 0,
            "num_prefix_steps_kept": 0,
            "num_prefix_steps_dropped": 0,
            "core_too_long": 0,
        }
        return text, info

    if task_type == "next_step":
        if smart_truncate_next_step_prefix:
            if tokenizer is None:
                raise ValueError("tokenizer is required for smart next-step prefix truncation")
            return format_next_step_with_recent_prefix(example, tokenizer, max_length=max_length)
        text = format_next_step_without_prefix_truncation(example)
        info = {
            "num_prefix_steps_original": len(example.get("prefix_steps", [])),
            "num_prefix_steps_kept": len(example.get("prefix_steps", [])),
            "num_prefix_steps_dropped": 0,
            "core_too_long": 0,
        }
        return text, info

    raise ValueError(f"Unknown task_type: {task_type}")


In [ ]:
assert abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) < 1e-8

groups = np.array([x["group_problem"] for x in all_examples])
indices = np.arange(len(all_examples))

gss1 = GroupShuffleSplit(n_splits=1, train_size=TRAIN_SIZE, random_state=SEED)
train_idx, temp_idx = next(gss1.split(indices, groups=groups))

temp_groups = groups[temp_idx]
relative_val_size = VAL_SIZE / (VAL_SIZE + TEST_SIZE)

gss2 = GroupShuffleSplit(n_splits=1, train_size=relative_val_size, random_state=SEED + 1)
val_rel_idx, test_rel_idx = next(gss2.split(temp_idx, groups=temp_groups))

val_idx = temp_idx[val_rel_idx]
test_idx = temp_idx[test_rel_idx]

train_examples = [all_examples[i] for i in train_idx]
val_examples = [all_examples[i] for i in val_idx]
test_examples = [all_examples[i] for i in test_idx]

def problem_set(examples):
    return set(x["group_problem"] for x in examples)

train_problems = problem_set(train_examples)
val_problems = problem_set(val_examples)
test_problems = problem_set(test_examples)

assert train_problems.isdisjoint(val_problems)
assert train_problems.isdisjoint(test_problems)
assert val_problems.isdisjoint(test_problems)

print("Rows:")
print("  train:", len(train_examples))
print("  val:  ", len(val_examples))
print("  test: ", len(test_examples))

print("\nUnique problems:")
print("  train:", len(train_problems))
print("  val:  ", len(val_problems))
print("  test: ", len(test_problems))

print("\nTask counts:")
print("  train:", Counter(x["task_type"] for x in train_examples))
print("  val:  ", Counter(x["task_type"] for x in val_examples))
print("  test: ", Counter(x["task_type"] for x in test_examples))

print("\nLabel counts:")
print("  train:", Counter(x["label"] for x in train_examples))
print("  val:  ", Counter(x["label"] for x in val_examples))
print("  test: ", Counter(x["label"] for x in test_examples))

Rows:
  train: 21704
  val:   2689
  test:  2768

Unique problems:
  train: 366
  val:   46
  test:  46

Task counts:
  train: Counter({'next_step': 20982, 'solution': 722})
  val:   Counter({'next_step': 2598, 'solution': 91})
  test:  Counter({'next_step': 2676, 'solution': 92})

Label counts:
  train: Counter({1: 16441, 0: 5263})
  val:   Counter({1: 2069, 0: 620})
  test:  Counter({1: 2124, 0: 644})


In [ ]:
def majority_accuracy(examples: List[Dict[str, Any]]) -> float:
    counts = Counter(x["label"] for x in examples)
    if not counts:
        return float("nan")
    return max(counts.values()) / sum(counts.values())

def task_majority_accuracy(examples: List[Dict[str, Any]]) -> Dict[str, float]:
    out = {}
    for task in sorted(set(x["task_type"] for x in examples)):
        task_examples = [x for x in examples if x["task_type"] == task]
        out[task] = majority_accuracy(task_examples)
    return out

print("Overall train majority baseline:", majority_accuracy(train_examples))
print("Overall val majority baseline:  ", majority_accuracy(val_examples))
print("Overall test majority baseline: ", majority_accuracy(test_examples))

print("\nTask-wise test majority baselines:")
print(task_majority_accuracy(test_examples))

Overall train majority baseline: 0.7575101363803907
Overall val majority baseline:   0.7694310152473038
Overall test majority baseline:  0.7673410404624278

Task-wise test majority baselines:
{'next_step': 0.7765321375186846, 'solution': 0.5}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

for ex in all_examples:
    text, info = format_example(
        ex,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
        smart_truncate_next_step_prefix=True,
    )
    ex["text"] = text
    ex.update(info)

def token_length_text(text: str) -> int:
    return len(tokenizer(text, truncation=False)["input_ids"])

def diagnostic_full_next_step_length(example: Dict[str, Any]) -> int:
    full_text = format_next_step_without_prefix_truncation(example)
    return token_length_text(full_text)

sample_for_lengths = random.sample(all_examples, min(len(all_examples), 5000))
lengths_after = np.array([token_length_text(x["text"]) for x in sample_for_lengths])

print("Token length summary after smart formatting:")
for q in [50, 75, 90, 95, 99, 100]:
    print(f"  p{q}: {np.percentile(lengths_after, q):.0f}")

print(f"\nMAX_LENGTH = {MAX_LENGTH}")
print(f"Estimated percent still over MAX_LENGTH after formatting: {(lengths_after > MAX_LENGTH).mean() * 100:.2f}%")

next_sample = [x for x in sample_for_lengths if x["task_type"] == "next_step"]
if next_sample:
    dropped_counts = np.array([x["num_prefix_steps_dropped"] for x in next_sample])
    core_too_long = np.array([x["core_too_long"] for x in next_sample])
    print("\nNext-step prefix truncation diagnostics on sample:")
    print(f"  Percent with at least one omitted early prefix step: {(dropped_counts > 0).mean() * 100:.2f}%")
    print(f"  Average omitted early prefix steps: {dropped_counts.mean():.2f}")
    print(f"  Percent where problem+target core already exceeded budget: {core_too_long.mean() * 100:.2f}%")

    full_lengths = np.array([diagnostic_full_next_step_length(x) for x in random.sample(next_sample, min(len(next_sample), 1000))])
    print(f"  Estimated percent over MAX_LENGTH before smart prefix truncation: {(full_lengths > MAX_LENGTH).mean() * 100:.2f}%")

print("\nExample formatted input:")
print(all_examples[0]["text"][:2500])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Token length summary after smart formatting:
  p50: 273
  p75: 376
  p90: 507
  p95: 605
  p99: 805
  p100: 1206

MAX_LENGTH = 2048
Estimated percent still over MAX_LENGTH after formatting: 0.00%

Next-step prefix truncation diagnostics on sample:
  Percent with at least one omitted early prefix step: 0.00%
  Average omitted early prefix steps: 0.00
  Percent where problem+target core already exceeded budget: 0.00%
  Estimated percent over MAX_LENGTH before smart prefix truncation: 0.00%

Example formatted input:
Instruction:
Verify whether the target step is valid given the problem and the previous steps.

Task: next-step verification

Target step:
I notice that there are three groups of people: Democrats, Republicans, and Independent.

Problem:
A Senate committee has 5 Democrats, 5 Republicans, and 1 Independent.  In how many ways can they sit around a circular table if all the members of each party all sit next to each other?  (Two seatings are considered equivalent if one is a rota

In [ ]:
def to_hf_dataset(examples: List[Dict[str, Any]]) -> Dataset:
    keep = []
    for x in examples:
        keep.append({
            "text": x["text"],
            "label": int(x["label"]),
        })
    return Dataset.from_list(keep)

train_dataset = to_hf_dataset(train_examples)
val_dataset = to_hf_dataset(val_examples)
test_dataset = to_hf_dataset(test_examples)

def tokenize_batch(batch):
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    tokenized.pop("token_type_ids", None)
    return tokenized

remove_cols = ["text"]

tokenized_train = train_dataset.map(tokenize_batch, batched=True, remove_columns=remove_cols)
tokenized_val = val_dataset.map(tokenize_batch, batched=True, remove_columns=remove_cols)
tokenized_test = test_dataset.map(tokenize_batch, batched=True, remove_columns=remove_cols)

print(tokenized_train)
print(tokenized_val)
print(tokenized_test)


Map:   0%|          | 0/21704 [00:00<?, ? examples/s]

Map:   0%|          | 0/2689 [00:00<?, ? examples/s]

Map:   0%|          | 0/2768 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 21704
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 2689
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 2768
})


In [ ]:
id2label = {0: "incorrect_or_invalid", 1: "correct_or_valid"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    problem_type="single_label_classification",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    attn_implementation="sdpa",
)

model.gradient_checkpointing_enable()

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Model loaded:", MODEL_NAME)
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: answerdotai/ModernBERT-large
Number of labels: 2
id2label: {0: 'incorrect_or_invalid', 1: 'correct_or_valid'}
Total parameters: 395,833,346
Trainable parameters: 395,833,346


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    labels = np.asarray(labels)
    preds = np.argmax(logits, axis=-1)

    logits_shifted = logits - logits.max(axis=-1, keepdims=True)
    probs = np.exp(logits_shifted) / np.exp(logits_shifted).sum(axis=-1, keepdims=True)
    prob_pos = probs[:, 1]

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0,
    )

    out = {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

    if len(np.unique(labels)) == 2:
        out["roc_auc"] = roc_auc_score(labels, prob_pos)
    else:
        out["roc_auc"] = float("nan")

    return out

def make_class_weights(examples: List[Dict[str, Any]]) -> torch.Tensor:
    counts = Counter(int(x["label"]) for x in examples)
    total = sum(counts.values())
    weights = []
    for cls in [0, 1]:
        if counts[cls] == 0:
            weights.append(1.0)
        else:
            weights.append(total / (2.0 * counts[cls]))
    return torch.tensor(weights, dtype=torch.float32)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

class_weights = make_class_weights(train_examples) if USE_CLASS_WEIGHTS else None
print("USE_CLASS_WEIGHTS:", USE_CLASS_WEIGHTS)
print("class_weights:", class_weights)

USE_CLASS_WEIGHTS: True
class_weights: tensor([2.0619, 0.6601])


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    bf16=torch.cuda.is_available(),
    tf32=torch.cuda.is_available(),
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="eval_balanced_accuracy",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
)

trainer_common_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

if USE_CLASS_WEIGHTS:
    trainer = WeightedTrainer(
        **trainer_common_kwargs,
        class_weights=class_weights,
    )
else:
    trainer = Trainer(**trainer_common_kwargs)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1,Roc Auc
1,4.600612,0.591059,0.754556,0.607810,0.815495,0.880135,0.846583,0.705570
2,4.258444,0.599153,0.752696,0.609425,0.816501,0.875302,0.844880,0.705187


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2714, training_loss=4.614014153983018, metrics={'train_runtime': 2107.0729, 'train_samples_per_second': 20.601, 'train_steps_per_second': 1.288, 'total_flos': 3.498727836206988e+16, 'train_loss': 4.614014153983018, 'epoch': 2.0})

In [ ]:
test_metrics = trainer.evaluate(tokenized_test)
print("Test metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

Training Loss,Validation Loss,Epoch,Accuracy,Balanced Accuracy,Precision,Recall,F1,Roc Auc
4.258444,0.604165,2,0.733020,0.576638,0.800173,0.869115,0.833220,0.685013


Test metrics:
eval_loss: 0.6041645407676697
eval_accuracy: 0.7330202312138728
eval_balanced_accuracy: 0.5766381841363418
eval_precision: 0.800173385348938
eval_recall: 0.8691148775894538
eval_f1: 0.8332204919882645
eval_roc_auc: 0.6850125305587722


In [ ]:
pred_output = trainer.predict(tokenized_test)
logits = pred_output.predictions
labels = np.array([int(x["label"]) for x in test_examples])
preds = np.argmax(logits, axis=-1)

logits_shifted = logits - logits.max(axis=-1, keepdims=True)
probs = np.exp(logits_shifted) / np.exp(logits_shifted).sum(axis=-1, keepdims=True)
prob_pos = probs[:, 1]

results_df = pd.DataFrame({
    "task_type": [x["task_type"] for x in test_examples],
    "label": labels,
    "pred": preds,
    "prob_correct_or_valid": prob_pos,
    "problem": [x["problem"] for x in test_examples],
})

def metrics_for_frame(df: pd.DataFrame) -> Dict[str, Any]:
    y_true = df["label"].to_numpy()
    y_pred = df["pred"].to_numpy()
    y_prob = df["prob_correct_or_valid"].to_numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    out = {
        "n": len(df),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "majority_baseline": max(Counter(y_true).values()) / len(y_true),
    }

    if len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        out["roc_auc"] = float("nan")

    return out

summary_rows = []
summary_rows.append({"split": "test", "task_type": "overall", **metrics_for_frame(results_df)})

for task in sorted(results_df["task_type"].unique()):
    task_df = results_df[results_df["task_type"] == task]
    summary_rows.append({"split": "test", "task_type": task, **metrics_for_frame(task_df)})

metrics_summary = pd.DataFrame(summary_rows)
display(metrics_summary)

print("\nOverall confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(results_df["label"], results_df["pred"]))

for task in sorted(results_df["task_type"].unique()):
    task_df = results_df[results_df["task_type"] == task]
    print(f"\n{task} confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(task_df["label"], task_df["pred"]))

metrics_summary_path = Path(OUTPUT_DIR) / "test_metrics_summary.csv"
results_path = Path(OUTPUT_DIR) / "test_predictions.csv"

metrics_summary.to_csv(metrics_summary_path, index=False)
results_df.to_csv(results_path, index=False)

print("\nSaved:")
print(metrics_summary_path)
print(results_path)

,split,task_type,n,accuracy,balanced_accuracy,precision,recall,f1,majority_baseline,roc_auc
0,test,overall,2768,0.733020,0.576638,0.800173,0.869115,0.833220,0.767341,0.685013
1,test,next_step,2676,0.730194,0.554130,0.798678,0.872474,0.833947,0.776532,0.665265
2,test,solution,92,0.815217,0.815217,0.891892,0.717391,0.795181,0.500000,0.933601



Overall confusion matrix [[TN, FP], [FN, TP]]:
[[ 183  461]
 [ 278 1846]]

next_step confusion matrix [[TN, FP], [FN, TP]]:
[[ 141  457]
 [ 265 1813]]

solution confusion matrix [[TN, FP], [FN, TP]]:
[[42  4]
 [13 33]]

Saved:
/content/modernbert_joint_verifier_outputs/test_metrics_summary.csv
/content/modernbert_joint_verifier_outputs/test_predictions.csv


In [ ]:
def show_examples(df: pd.DataFrame, kind: str, n: int = 3):
    if kind == "false_positive":
        sub = df[(df["label"] == 0) & (df["pred"] == 1)]
    elif kind == "false_negative":
        sub = df[(df["label"] == 1) & (df["pred"] == 0)]
    else:
        raise ValueError("kind must be false_positive or false_negative")

    print(f"\n{kind}: showing {min(n, len(sub))} of {len(sub)}")
    for idx in sub.head(n).index:
        ex = test_examples[idx]
        print("=" * 100)
        print("task:", ex["task_type"], "label:", ex["label"], "pred:", int(df.loc[idx, "pred"]),
              "prob_valid:", float(df.loc[idx, "prob_correct_or_valid"]))
        print(ex["text"][:2500])

show_examples(results_df, "false_positive", n=2)
show_examples(results_df, "false_negative", n=2)


false_positive: showing 2 of 461
task: next_step label: 0 pred: 1 prob_valid: 0.8766343593597412
Instruction:
Verify whether the target step is valid given the problem and the previous steps.

Task: next-step verification

Target step:
Also, since $a$ and $b$ are positive, they must be factors of $2009$.

Problem:
Positive integers $a$, $b$, and $2009$, with $a<b<2009$, form a geometric sequence with an integer ratio. What is $a$?

Previous steps, with the most recent available context preserved:
1. To form a geometric sequence, the ratio of consecutive terms must be constant.
2. That is, $\frac{b}{a}=\frac{2009}{b}$.
3. I can cross-multiply this equation to get $b^2=2009a$.
4. Since $b$ is an integer, $b^2$ must be a perfect square.

Binary label meaning: 1 = valid/correct step, 0 = invalid/incorrect step.
task: next_step label: 0 pred: 1 prob_valid: 0.83601975440979
Instruction:
Verify whether the target step is valid given the problem and the previous steps.

Task: next-step verifi

In [ ]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

metadata = {
    "base_model": MODEL_NAME,
    "save_dir": SAVE_DIR,
    "max_length": MAX_LENGTH,
    "next_step_prefix_truncation": {
        "strategy": "manual_recent_prefix_budget",
        "description": "For next-step verification, preserve instruction/task/target/problem and omit earliest prefix steps first.",
        "safety_tokens": PREFIX_TRUNCATION_SAFETY_TOKENS,
        "train_next_step_examples_with_dropped_prefix": int(sum((x["task_type"] == "next_step" and x.get("num_prefix_steps_dropped", 0) > 0) for x in train_examples)),
        "val_next_step_examples_with_dropped_prefix": int(sum((x["task_type"] == "next_step" and x.get("num_prefix_steps_dropped", 0) > 0) for x in val_examples)),
        "test_next_step_examples_with_dropped_prefix": int(sum((x["task_type"] == "next_step" and x.get("num_prefix_steps_dropped", 0) > 0) for x in test_examples)),
    },
    "id2label": id2label,
    "label2id": label2id,
    "train_size_rows": len(train_examples),
    "val_size_rows": len(val_examples),
    "test_size_rows": len(test_examples),
    "train_unique_problems": len(train_problems),
    "val_unique_problems": len(val_problems),
    "test_unique_problems": len(test_problems),
    "use_class_weights": USE_CLASS_WEIGHTS,
    "class_weights": class_weights.tolist() if class_weights is not None else None,
    "training_hyperparameters": {
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    },
    "test_metrics": {k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float, np.floating))},
}

with open(Path(SAVE_DIR) / "verifier_training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

metrics_summary.to_csv(Path(SAVE_DIR) / "test_metrics_summary.csv", index=False)
results_df.to_csv(Path(SAVE_DIR) / "test_predictions.csv", index=False)

print("Saved best model and tokenizer to:", SAVE_DIR)
print("Files:")
for p in sorted(Path(SAVE_DIR).iterdir()):
    print(" -", p.name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model and tokenizer to: /content/modernbert_joint_verifier_best
Files:
 - config.json
 - model.safetensors
 - test_metrics_summary.csv
 - test_predictions.csv
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin
 - verifier_training_metadata.json


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

reward_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
reward_model = AutoModelForSequenceClassification.from_pretrained(
    SAVE_DIR,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    attn_implementation="sdpa",
)
reward_model.eval()

if torch.cuda.is_available():
    reward_model = reward_model.to("cuda")

def score_texts(texts: List[str], batch_size: int = 4) -> List[float]:
    scores = []
    device = next(reward_model.parameters()).device

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        inputs = reward_tokenizer(
            batch_texts,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        )
        inputs.pop("token_type_ids", None)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = reward_model(**inputs).logits
            probs = torch.softmax(logits.float(), dim=-1)[:, 1]

        scores.extend(probs.detach().cpu().tolist())

    return scores

def format_next_step_for_reward(problem: str, prefix_steps: List[str], target_step: str) -> str:
    ex = {
        "task_type": "next_step",
        "instruction": "Verify whether the target step is mathematically valid given the problem and the previous steps.",
        "problem": problem,
        "prefix_steps": prefix_steps,
        "target_step": target_step,
        "solution": "",
    }
    text, _ = format_example(
        ex,
        tokenizer=reward_tokenizer,
        max_length=MAX_LENGTH,
        smart_truncate_next_step_prefix=True,
    )
    return text

def format_solution_for_reward(problem: str, solution: str) -> str:
    ex = {
        "task_type": "solution",
        "instruction": "Verify whether the following full solution is correct.",
        "problem": problem,
        "prefix_steps": [],
        "target_step": "",
        "solution": solution,
    }
    text, _ = format_example(
        ex,
        tokenizer=reward_tokenizer,
        max_length=MAX_LENGTH,
        smart_truncate_next_step_prefix=True,
    )
    return text

smoke_text = test_examples[0]["text"]
smoke_score = score_texts([smoke_text])[0]
print("Smoke-test P(label=1):", smoke_score)
print("True label:", test_examples[0]["label"])


Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

Smoke-test P(label=1): 0.9773707389831543
True label: 1


In [ ]:
!cd /content && zip -r modernbert_joint_verifier_best.zip modernbert_joint_verifier_best >/dev/null

from google.colab import files
files.download("/content/modernbert_joint_verifier_best.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>